In [0]:
%run "/Users/patelrahul2614@gmail.com/databrick_demo/Digital_Banking_LakeHouse_Capstone/includes"

In [0]:
# Read the catalog name from the widget
_catalog = dbutils.widgets.get("catalog")

# Read the branch table from the Bronze layer

bronze_table = f"{_catalog}.bronze.branches"
df_bronze = spark.table(bronze_table)

# Drop metadata
metadata_cols = [
    "file_name", "file_path", "ingestion_date"
]
df_clean = df_bronze.drop(*[c for c in metadata_cols if c in df_bronze.columns])

# Drop duplicates on branch_id

df_clean = df_clean.dropDuplicates(["branch_id"])

# Simple validation: key columns should not be null
non_null_cols = ["branch_id", "branch_code", "branch_name", "city", "state", "branch_status"]

# Only validate columns that actually exist in the dataframe
non_null_cols = [c for c in non_null_cols if c in df_clean.columns]

for c in non_null_cols:
    df_clean = df_clean.filter(col(c).isNotNull())

# Write the cleaned data to the Silver layer
silver_table = f"{_catalog}.silver.silver_branches"
df_clean.write.mode("overwrite").saveAsTable(silver_table)
